In [1]:
import numpy as np
from scipy.stats import pearsonr
from tqdm import tqdm


def calc_r_rmse_maps_like_old_4d(predicty, testy, show_progress=False):
    """
    完全按照你原来的思路改成 4 维：
    - 对每个格点 (level, lat, lon)，沿 time 计算 pearsonr
    - 对每个格点 (level, lat, lon)，沿 time 计算 RMSE

    输入
    ----
    predicty, testy : np.ndarray
        shape = (time, level, lat, lon)

    输出
    ----
    r_map, rmse_map : np.ndarray
        shape = (level, lat, lon)
    """
    predicty = np.asarray(predicty, dtype=np.float64)
    testy = np.asarray(testy, dtype=np.float64)

    if predicty.shape != testy.shape:
        raise ValueError(f"shape 不一致: {predicty.shape} vs {testy.shape}")
    if predicty.ndim != 4:
        raise ValueError("输入必须是四维 (time, level, lat, lon)")

    _, nlevel, nlat, nlon = predicty.shape

    r_map = np.full((nlevel, nlat, nlon), np.nan, dtype=np.float64)
    rmse_map = np.full((nlevel, nlat, nlon), np.nan, dtype=np.float64)

    level_iter = range(nlevel)
    if show_progress:
        level_iter = tqdm(level_iter, desc="Calculating R/RMSE maps")

    for k in level_iter:
        for i in range(nlat):
            for j in range(nlon):
                x = predicty[:, k, i, j]
                y = testy[:, k, i, j]

                # 与你旧代码保持一致：
                # 只要该格点时间序列中存在 NaN，就直接给 NaN
                if np.isnan(x).any() or np.isnan(y).any():
                    r_map[k, i, j] = np.nan
                    rmse_map[k, i, j] = np.nan
                    continue

                # R
                try:
                    r_map[k, i, j], _ = pearsonr(x, y)
                except:
                    r_map[k, i, j] = np.nan

                # RMSE
                try:
                    rmse_map[k, i, j] = np.sqrt(np.mean((x - y) ** 2))
                except:
                    rmse_map[k, i, j] = np.nan

    return r_map, rmse_map


def _sample_block_indices(T, block_size, rng):
    """
    moving block bootstrap:
    每次随机抽连续块，直到拼满长度 T
    """
    if block_size < 1:
        raise ValueError("block_size 必须 >= 1")
    block_size = min(block_size, T)

    idx = []
    max_start = T - block_size

    while len(idx) < T:
        start = 0 if max_start <= 0 else rng.integers(0, max_start + 1)
        idx.extend(range(start, start + block_size))

    return np.array(idx[:T], dtype=int)


def bootstrap_ci_like_old_r_rmse_4d(
    predicty,
    testy,
    n_boot=1000,
    block_size=5,
    ci=95,
    random_state=42,
    show_progress=True
):
    """
    按你原来的 R 定义做 block bootstrap 95% CI（4维版）

    统计量定义：
    - R_overall    = nanmean(每个(level,lat,lon)格点沿time算出来的R图)
    - RMSE_overall = nanmean(每个(level,lat,lon)格点沿time算出来的RMSE图)

    输入
    ----
    predicty, testy : np.ndarray
        shape = (time, level, lat, lon)

    返回
    ----
    {
        "r_map": 原始R图,            shape = (level, lat, lon)
        "rmse_map": 原始RMSE图,      shape = (level, lat, lon)
        "R_mean": {
            "estimate": ...,
            "ci_lower": ...,
            "ci_upper": ...,
            "ci": (..., ...)
        },
        "RMSE_mean": {
            "estimate": ...,
            "ci_lower": ...,
            "ci_upper": ...,
            "ci": (..., ...)
        },
        "R_bootstrap": ...,
        "RMSE_bootstrap": ...
    }
    """
    predicty = np.asarray(predicty, dtype=np.float64)
    testy = np.asarray(testy, dtype=np.float64)

    if predicty.shape != testy.shape:
        raise ValueError(f"shape 不一致: {predicty.shape} vs {testy.shape}")
    if predicty.ndim != 4:
        raise ValueError("输入必须是四维 (time, level, lat, lon)")

    T = predicty.shape[0]
    rng = np.random.default_rng(random_state)
    alpha = (100 - ci) / 2

    # 原始点估计（与你旧代码一致）
    r_map, rmse_map = calc_r_rmse_maps_like_old_4d(predicty, testy, show_progress=False)
    R_mean_est = np.nanmean(r_map)
    RMSE_mean_est = np.nanmean(rmse_map)

    # bootstrap
    R_boot = np.full(n_boot, np.nan, dtype=np.float64)
    RMSE_boot = np.full(n_boot, np.nan, dtype=np.float64)

    boot_iter = range(n_boot)
    if show_progress:
        boot_iter = tqdm(boot_iter, desc="Bootstrap")

    for b in boot_iter:
        idx = _sample_block_indices(T, block_size, rng)

        pred_b = predicty[idx, :, :, :]
        test_b = testy[idx, :, :, :]

        r_map_b, rmse_map_b = calc_r_rmse_maps_like_old_4d(pred_b, test_b, show_progress=False)

        R_boot[b] = np.nanmean(r_map_b)
        RMSE_boot[b] = np.nanmean(rmse_map_b)

    # 去掉无效值
    R_boot_valid = R_boot[np.isfinite(R_boot)]
    RMSE_boot_valid = RMSE_boot[np.isfinite(RMSE_boot)]

    R_ci = (
        np.percentile(R_boot_valid, alpha),
        np.percentile(R_boot_valid, 100 - alpha)
    ) if len(R_boot_valid) > 0 else (np.nan, np.nan)

    RMSE_ci = (
        np.percentile(RMSE_boot_valid, alpha),
        np.percentile(RMSE_boot_valid, 100 - alpha)
    ) if len(RMSE_boot_valid) > 0 else (np.nan, np.nan)

    return {
        "r_map": r_map,
        "rmse_map": rmse_map,
        "R_mean": {
            "estimate": float(R_mean_est),
            "ci_lower": float(R_ci[0]),
            "ci_upper": float(R_ci[1]),
            "ci": (float(R_ci[0]), float(R_ci[1]))
        },
        "RMSE_mean": {
            "estimate": float(RMSE_mean_est),
            "ci_lower": float(RMSE_ci[0]),
            "ci_upper": float(RMSE_ci[1]),
            "ci": (float(RMSE_ci[0]), float(RMSE_ci[1]))
        },
        "R_bootstrap": R_boot_valid,
        "RMSE_bootstrap": RMSE_boot_valid
    }

In [2]:
import Auto_paint_self
cnn_transformer_fusion_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_1_5year_order_fusion.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_1_5year_order_fusion.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_1_5year_order_fusion.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_2_5year_order_fusion.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_2_5year_order_fusion.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_2_5year_order_fusion.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_3_5year_order_fusion.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_3_5year_order_fusion.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_3_5year_order_fusion.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_4_5year_order_fusion.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_4_5year_order_fusion.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_4_5year_order_fusion.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_5_5year_order_fusion.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_5_5year_order_fusion.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_5_5year_order_fusion.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_6_5year_order_fusion.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_6_5year_order_fusion.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_6_5year_order_fusion.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
cnn_transformer_fusion_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_1_5year_order_fusion.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_1_5year_order_fusion.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_1_5year_order_fusion.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_2_5year_order_fusion.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_2_5year_order_fusion.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_2_5year_order_fusion.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_3_5year_order_fusion.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_3_5year_order_fusion.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_3_5year_order_fusion.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_4_5year_order_fusion.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_4_5year_order_fusion.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_4_5year_order_fusion.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_5_5year_order_fusion.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_5_5year_order_fusion.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_5_5year_order_fusion.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_fusion_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_6_5year_order_fusion.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_6_5year_order_fusion.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_fusion_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_6_5year_order_fusion.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [3]:
import Auto_paint_self
cnn_transformer_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_1_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_1_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_1_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_2_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_2_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_2_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_3_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_3_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_3_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_4_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_4_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_4_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_5_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_5_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_5_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_test_u_6_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_v_6_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_testy_w_6_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
cnn_transformer_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_1_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_1_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_1_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_2_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_2_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_2_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_3_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_3_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_3_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_4_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_4_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_4_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_5_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_5_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_5_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

cnn_transformer_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_u_6_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_v_6_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/cnn_transformer_predicty_w_6_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [15]:
import Auto_paint_self
transformer_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_test_u_1_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_v_1_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_w_1_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_test_u_2_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_v_2_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_w_2_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_test_u_3_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_v_3_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_w_3_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_test_u_4_5year_orders.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_v_4_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_w_4_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_test_u_5_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_v_5_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_w_5_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_test_u_6_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_v_6_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_testy_w_6_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
transformer_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_u_1_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_v_1_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_w_1_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_u_2_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_v_2_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_w_2_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_u_3_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_v_3_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_w_3_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_u_4_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_v_4_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_w_4_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_u_5_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_v_5_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_w_5_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

transformer_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_u_6_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_v_6_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/transformer_predicty_w_6_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [7]:
import Auto_paint_self
ann_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_1_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_1_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_1_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_2_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_2_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_2_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_3_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_3_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_3_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_4_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_4_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_4_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_5_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_5_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_5_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_6_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_6_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_6_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
ann_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_1_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_1_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_1_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_2_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_2_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_2_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_3_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_3_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_3_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_4_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_4_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_4_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_5_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_5_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_5_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_6_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_6_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_6_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [16]:
import Auto_paint_self
unet_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_test_u_1_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_v_1_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_w_1_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_test_u_2_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_v_2_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_w_2_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_test_u_3_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_v_3_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_w_3_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_test_u_4_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_v_4_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_w_4_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_test_u_5_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_v_5_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_w_5_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_test_u_6_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_v_6_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_testy_w_6_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
unet_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_u_1_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_v_1_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_w_1_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_u_2_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_v_2_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_w_2_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_u_3_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_v_3_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_w_3_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_u_4_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_v_4_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_w_4_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_u_5_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_v_5_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_w_5_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

unet_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_u_6_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_v_6_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/unet_predicty_w_6_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [8]:
import Auto_paint_self
ann_lr0000001_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_1_lr0000001_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_1_lr0000001_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_1_lr0000001_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_2_lr0000001_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_2_lr0000001_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_2_lr0000001_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_3_lr0000001_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_3_lr0000001_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_3_lr0000001_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_4_lr0000001_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_4_lr0000001_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_4_lr0000001_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_5_lr0000001_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_5_lr0000001_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_5_lr0000001_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_6_lr0000001_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_6_lr0000001_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_6_lr0000001_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
ann_lr0000001_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_1_lr0000001_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_1_lr0000001_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_1_lr0000001_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_2_lr0000001_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_2_lr0000001_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_2_lr0000001_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_3_lr0000001_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_3_lr0000001_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_3_lr0000001_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_4_lr0000001_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_4_lr0000001_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_4_lr0000001_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_5_lr0000001_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_5_lr0000001_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_5_lr0000001_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_lr0000001_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_6_lr0000001_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_6_lr0000001_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_lr0000001_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_6_lr0000001_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [9]:
import Auto_paint_self
ann_xianyan_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_u_1_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_v_1_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_w_1_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_u_2_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_v_2_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_w_2_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_u_3_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_v_3_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_w_3_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_u_4_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_v_4_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_w_4_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_u_5_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_v_5_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_w_5_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_u_6_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_v_6_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_xianyan_w_6_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
ann_xianyan_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_u_1_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_v_1_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_w_1_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_u_2_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_v_2_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_w_2_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_u_3_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_v_3_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_w_3_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_u_4_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_v_4_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_w_4_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_u_5_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_v_5_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_w_5_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_xianyan_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_u_6_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_v_6_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_xianyan_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_xianyan_w_6_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [10]:
import Auto_paint_self
ann_pe_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_1_PE_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_1_PE_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_1_PE_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_2_PE_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_2_PE_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_2_PE_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_3_PE_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_3_PE_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_3_PE_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_4_PE_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_4_PE_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_4_PE_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_5_PE_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_5_PE_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_5_PE_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_6_PE_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_6_PE_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_6_PE_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
ann_pe_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_1_PE_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_1_PE_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_1_PE_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_2_PE_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_2_PE_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_2_PE_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_3_PE_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_3_PE_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_3_PE_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_4_PE_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_4_PE_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_4_PE_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_5_PE_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_5_PE_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_5_PE_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_pe_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_6_PE_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_6_PE_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_pe_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_6_PE_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [11]:
import Auto_paint_self
ann_6time_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_1_6time_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_1_6time_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_1_6time_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_2_6time_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_2_6time_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_2_6time_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_3_6time_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_3_6time_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_3_6time_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_4_6time_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_4_6time_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_4_6time_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_5_6time_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_5_6time_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_5_6time_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_6_6time_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_6_6time_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_6_6time_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
ann_6time_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_1_6time_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_1_6time_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_1_6time_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_2_6time_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_2_6time_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_2_6time_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_3_6time_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_3_6time_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_3_6time_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_4_6time_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_4_6time_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_4_6time_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_5_6time_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_5_6time_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_5_6time_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_6time_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_6_6time_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_6_6time_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_6time_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_6_6time_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [12]:
import Auto_paint_self
ann_onlyu_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_1_onlyu_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_1_onlyv_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_1_onlyw_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_2_onlyu_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_2_onlyv_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_2_onlyw_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_3_onlyu_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_3_onlyv_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_3_onlyw_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_4_onlyu_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_4_onlyv_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_4_onlyw_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_5_onlyu_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_5_onlyv_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_5_onlyw_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_6_onlyu_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_6_onlyv_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_6_onlyw_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
ann_onlyu_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_1_onlyu_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_1_onlyv_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_1_onlyw_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_2_onlyu_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_2_onlyv_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_2_onlyw_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_3_onlyu_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_3_onlyv_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_3_onlyw_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_4_onlyu_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_4_onlyv_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_4_onlyw_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_5_onlyu_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_5_onlyv_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_5_onlyw_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_onlyu_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_6_onlyu_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyv_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_6_onlyv_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_onlyw_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_6_onlyw_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [17]:
import Auto_paint_self
ann_cdf_testy_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_1_cdf_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_1_cdf_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_1_cdf_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_testy_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_2_cdf_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_2_cdf_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_2_cdf_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_testy_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_3_cdf_5year_order.nc','testy_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_3_cdf_5year_order.nc','testy_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_3_cdf_5year_order.nc','testy_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_testy_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_4_cdf_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_4_cdf_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_4_cdf_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_testy_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_5_cdf_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_5_cdf_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_5_cdf_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_testy_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_u_6_cdf_5year_order.nc','testy_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_v_6_cdf_5year_order.nc','testy_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_testy_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_testy_w_6_cdf_5year_order.nc','testy_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

import Auto_paint_self
ann_cdf_predicty_u_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_1_cdf_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_v_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_1_cdf_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_w_1,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_1_cdf_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_predicty_u_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_2_cdf_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_v_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_2_cdf_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_w_2,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_2_cdf_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_predicty_u_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_3_cdf_5year_order.nc','predicty_u','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_v_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_3_cdf_5year_order.nc','predicty_v','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_w_3,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_3_cdf_5year_order.nc','predicty_w','yes','time','2023-01-01-00','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_predicty_u_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_4_cdf_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_v_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_4_cdf_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_w_4,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_4_cdf_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_predicty_u_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_5_cdf_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_v_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_5_cdf_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_w_5,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_5_cdf_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

ann_cdf_predicty_u_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_u_6_cdf_5year_order.nc','predicty_u','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_v_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_v_6_cdf_5year_order.nc','predicty_v','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ann_cdf_predicty_w_6,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one','E:/huawei/result/ann_predicty_w_6_cdf_5year_order.nc','predicty_w','yes','time','2023-01-01-01','2023-12-31-23','yes','longitude','yes','latitude',20.0,25.5,109.25,117.25,0.25,0.25,'no','all','level','all',changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

In [22]:
import numpy as np
from tqdm import tqdm

# =========================================================
# 0) 参数
# =========================================================
N_BOOT = 1000        # 正式跑建议 1000；先测试可改成 200
BLOCK_SIZE = 5       # moving block bootstrap
BATCH_BOOT = 16      # 每批处理多少个 bootstrap replicate；内存紧张可改 8
BASE_SEED = 42
SHOW_BOOT_BAR = False   # True 会显示更细的 bootstrap 进度，False 只显示模型/组合进度

COMPONENTS = ["u", "v", "w"]
LAGS = [1, 2, 3, 4, 5, 6]

MODEL_CONFIG = [
    ("Origin", {"u": "ann", "v": "ann", "w": "ann"}),
    ("Prior", {"u": "ann_xianyan", "v": "ann_xianyan", "w": "ann_xianyan"}),
    ("Only", {"u": "ann_onlyu", "v": "ann_onlyv", "w": "ann_onlyw"}),
    ("P_E", {"u": "ann_pe", "v": "ann_pe", "w": "ann_pe"}),
    ("Lr_0.000001", {"u": "ann_lr0000001", "v": "ann_lr0000001", "w": "ann_lr0000001"}),
    ("6_times", {"u": "ann_6time", "v": "ann_6time", "w": "ann_6time"}),
    ("CDF", {"u": "ann_cdf", "v": "ann_cdf", "w": "ann_cdf"}),
    ("UNet", {"u": "unet", "v": "unet", "w": "unet"}),
    ("CNN-Transformer", {"u": "cnn_transformer", "v": "cnn_transformer", "w": "cnn_transformer"}),
    ("Transformer", {"u": "transformer", "v": "transformer", "w": "transformer"}),
    ("Fusion", {"u": "cnn_transformer_fusion", "v": "cnn_transformer_fusion", "w": "cnn_transformer_fusion"}),
]


# =========================================================
# 1) 工具函数：从命名空间抓变量
# =========================================================
def get_var_from_namespace(var_name, namespace):
    if var_name not in namespace:
        raise KeyError(f"没有找到变量: {var_name}")
    return namespace[var_name]


# =========================================================
# 2) moving block bootstrap
#    先生成索引，再转成 count matrix
# =========================================================
def sample_block_indices(T, block_size, rng):
    if block_size < 1:
        raise ValueError("block_size 必须 >= 1")

    block_size = min(block_size, T)
    idx = []
    max_start = T - block_size

    while len(idx) < T:
        start = 0 if max_start <= 0 else rng.integers(0, max_start + 1)
        idx.extend(range(start, start + block_size))

    return np.asarray(idx[:T], dtype=np.int32)


def build_bootstrap_count_matrix(T, n_boot, block_size, seed):
    """
    返回 count matrix: shape = (n_boot, T)
    第 b 行表示第 b 个 bootstrap replicate 中，每个时间位置被抽中的次数
    """
    rng = np.random.default_rng(seed)
    count_mat = np.zeros((n_boot, T), dtype=np.float32)

    for b in range(n_boot):
        idx = sample_block_indices(T, block_size, rng)
        count_mat[b, :] = np.bincount(idx, minlength=T).astype(np.float32)

    return count_mat


# =========================================================
# 3) 预处理：4D -> 2D，并剔除 time 维上有 NaN 的格点
# =========================================================
def prepare_pair_flat(predicty, testy):
    predicty = np.asarray(predicty, dtype=np.float32)
    testy = np.asarray(testy, dtype=np.float32)

    if predicty.shape != testy.shape:
        raise ValueError(f"shape 不一致: {predicty.shape} vs {testy.shape}")
    if predicty.ndim != 4:
        raise ValueError(f"输入必须是 4 维 (time, level, lat, lon)，当前 predicty.ndim = {predicty.ndim}")

    T = predicty.shape[0]

    pred2 = predicty.reshape(T, -1)
    test2 = testy.reshape(T, -1)

    # 旧逻辑：只要该格点 time 序列中 pred 或 test 任一存在 NaN，则该格点直接剔除
    valid = ~(np.isnan(pred2).any(axis=0) | np.isnan(test2).any(axis=0))
    pred_valid = pred2[:, valid]
    test_valid = test2[:, valid]

    if pred_valid.shape[1] == 0:
        raise ValueError("该 pred/test 对在去除 NaN 后没有任何有效格点。")

    return pred_valid, test_valid


# =========================================================
# 4) 单个 pair 的原始点估计（不 bootstrap）
# =========================================================
def calc_pair_point_estimate_from_flat(pred_flat, test_flat):
    pred_flat = np.asarray(pred_flat, dtype=np.float32)
    test_flat = np.asarray(test_flat, dtype=np.float32)

    n = pred_flat.shape[0]

    # 预先构造
    x = pred_flat
    y = test_flat
    x2 = x * x
    y2 = y * y
    xy = x * y
    err2 = (x - y) * (x - y)

    # 各像素统计量
    Sx = np.sum(x, axis=0, dtype=np.float64)
    Sy = np.sum(y, axis=0, dtype=np.float64)
    Sx2 = np.sum(x2, axis=0, dtype=np.float64)
    Sy2 = np.sum(y2, axis=0, dtype=np.float64)
    Sxy = np.sum(xy, axis=0, dtype=np.float64)
    Serr2 = np.sum(err2, axis=0, dtype=np.float64)

    # RMSE
    rmse_pix = np.sqrt(Serr2 / n)
    rmse_mean = float(np.mean(rmse_pix))

    # R（与 Pearson 定义一致）
    cov_num = Sxy - (Sx * Sy) / n
    varx_num = Sx2 - (Sx * Sx) / n
    vary_num = Sy2 - (Sy * Sy) / n
    denom = np.sqrt(varx_num * vary_num)

    good = np.isfinite(denom) & (denom > 0)
    r_pix = cov_num[good] / denom[good]
    r_mean = float(np.mean(r_pix)) if r_pix.size > 0 else np.nan

    return r_mean, rmse_mean


# =========================================================
# 5) 单个 pair 的 bootstrap（快速版）
#    核心思路：不构造 (bs,T,P) 的 xb/yb，
#    而是用 count_mat @ X 做加权求和
# =========================================================
def bootstrap_pair_from_flat_fast(
    pred_flat,
    test_flat,
    count_mat,
    batch_boot=16,
    show_progress=False,
    desc="bootstrap"
):
    """
    pred_flat, test_flat: (T, P)
    count_mat: (n_boot, T)
    返回:
      r_boot:    (n_boot,)
      rmse_boot: (n_boot,)
    """
    pred_flat = np.asarray(pred_flat, dtype=np.float32)
    test_flat = np.asarray(test_flat, dtype=np.float32)
    count_mat = np.asarray(count_mat, dtype=np.float32)

    T, P = pred_flat.shape
    n_boot = count_mat.shape[0]
    n = float(T)

    # 预先准备好 time × pixel 基础数组
    X = pred_flat
    Y = test_flat
    X2 = X * X
    Y2 = Y * Y
    XY = X * Y
    ERR2 = (X - Y) * (X - Y)

    r_boot = np.full(n_boot, np.nan, dtype=np.float64)
    rmse_boot = np.full(n_boot, np.nan, dtype=np.float64)

    starts = list(range(0, n_boot, batch_boot))
    iterator = starts
    if show_progress:
        iterator = tqdm(starts, desc=desc, leave=False)

    for start in iterator:
        end = min(start + batch_boot, n_boot)
        W = count_mat[start:end, :]   # (bs, T)

        # 矩阵乘法：一次性得到每个 bootstrap replicate、每个像素的加权和
        Sx = W @ X
        Sy = W @ Y
        Sx2 = W @ X2
        Sy2 = W @ Y2
        Sxy = W @ XY
        Serr2 = W @ ERR2

        # RMSE
        rmse_pix = np.sqrt(Serr2 / n)          # (bs, P)
        rmse_boot[start:end] = np.mean(rmse_pix, axis=1)

        # R
        cov_num = Sxy - (Sx * Sy) / n
        varx_num = Sx2 - (Sx * Sx) / n
        vary_num = Sy2 - (Sy * Sy) / n
        denom = np.sqrt(varx_num * vary_num)

        with np.errstate(divide="ignore", invalid="ignore"):
            r_pix = cov_num / denom

        r_pix[~np.isfinite(r_pix)] = np.nan
        r_boot[start:end] = np.nanmean(r_pix, axis=1)

    return r_boot, rmse_boot


# =========================================================
# 6) 主程序
#    - 允许同一模型内不同 pair 的 T 不同
#    - 相同 T 复用 count matrix
#    - 模型层、组合层显示进度条
# =========================================================
namespace = globals()
all_results = {}

model_iter = tqdm(MODEL_CONFIG, desc="All models", dynamic_ncols=True, ascii=True)

for model_idx, (model_name, prefix_map) in enumerate(model_iter):
    print("=" * 90)
    print(f"Model: {model_name}")

    combo_list = [(comp, lag) for comp in COMPONENTS for lag in LAGS]

    # ---------- 先预处理 18 个组合 ----------
    flat_pairs = []
    point_r_list = []
    point_rmse_list = []

    prepare_bar = tqdm(
        combo_list,
        desc=f"{model_name} prepare",
        leave=False,
        dynamic_ncols=True,
        ascii=True
    )

    for comp, lag in prepare_bar:
        prefix = prefix_map[comp]

        pred_name = f"{prefix}_predicty_{comp}_{lag}"
        test_name = f"{prefix}_testy_{comp}_{lag}"

        pred = get_var_from_namespace(pred_name, namespace)
        test = get_var_from_namespace(test_name, namespace)

        pred_flat, test_flat = prepare_pair_flat(pred, test)
        flat_pairs.append((comp, lag, pred_flat, test_flat))

        r_est, rmse_est = calc_pair_point_estimate_from_flat(pred_flat, test_flat)
        point_r_list.append(r_est)
        point_rmse_list.append(rmse_est)

    # ---------- 模型原始点估计：18 个组合平均 ----------
    model_R_est = float(np.mean(point_r_list))
    model_RMSE_est = float(np.mean(point_rmse_list))

    # ---------- 为不同 T 建立 bootstrap count matrix 缓存 ----------
    count_cache = {}
    for _, _, pred_flat, _ in flat_pairs:
        T_here = pred_flat.shape[0]
        if T_here not in count_cache:
            seed_here = BASE_SEED + model_idx * 10000 + T_here
            count_cache[T_here] = build_bootstrap_count_matrix(
                T=T_here,
                n_boot=N_BOOT,
                block_size=BLOCK_SIZE,
                seed=seed_here
            )

    # ---------- 模型 bootstrap 累加 ----------
    model_R_boot_sum = np.zeros(N_BOOT, dtype=np.float64)
    model_RMSE_boot_sum = np.zeros(N_BOOT, dtype=np.float64)

    pair_bar = tqdm(
        flat_pairs,
        desc=f"{model_name} pairs",
        leave=False,
        dynamic_ncols=True,
        ascii=True
    )

    for comp, lag, pred_flat, test_flat in pair_bar:
        pair_bar.set_postfix_str(f"{comp}_{lag}")

        T_here = pred_flat.shape[0]
        count_mat = count_cache[T_here]

        r_boot, rmse_boot = bootstrap_pair_from_flat_fast(
            pred_flat=pred_flat,
            test_flat=test_flat,
            count_mat=count_mat,
            batch_boot=BATCH_BOOT,
            show_progress=SHOW_BOOT_BAR,
            desc=f"{model_name} {comp}_{lag} bootstrap"
        )

        model_R_boot_sum += r_boot
        model_RMSE_boot_sum += rmse_boot

    model_R_boot = model_R_boot_sum / len(flat_pairs)
    model_RMSE_boot = model_RMSE_boot_sum / len(flat_pairs)

    R_ci_lower, R_ci_upper = np.percentile(
        model_R_boot[np.isfinite(model_R_boot)], [2.5, 97.5]
    )
    RMSE_ci_lower, RMSE_ci_upper = np.percentile(
        model_RMSE_boot[np.isfinite(model_RMSE_boot)], [2.5, 97.5]
    )

    all_results[model_name] = {
        "R_est": model_R_est,
        "R_ci": (float(R_ci_lower), float(R_ci_upper)),
        "RMSE_est": model_RMSE_est,
        "RMSE_ci": (float(RMSE_ci_lower), float(RMSE_ci_upper)),
    }

    print(f"R     = {model_R_est:.6f} (95% CI: {R_ci_lower:.6f}, {R_ci_upper:.6f})")
    print(f"RMSE  = {model_RMSE_est:.6f} (95% CI: {RMSE_ci_lower:.6f}, {RMSE_ci_upper:.6f})")

print("\n" + "=" * 90)
print("Final summary")
print("=" * 90)

for model_name, res in all_results.items():
    print(
        f"{model_name:<18s} | "
        f"R = {res['R_est']:.6f} "
        f"[{res['R_ci'][0]:.6f}, {res['R_ci'][1]:.6f}] | "
        f"RMSE = {res['RMSE_est']:.6f} "
        f"[{res['RMSE_ci'][0]:.6f}, {res['RMSE_ci'][1]:.6f}]"
    )


CNN-Transformer prepare: 100%|#########################################################| 18/18 [01:56<00:00,  4.64s/it]
                                                                                                                       
All models:  82%|#######################################################6            | 9/11 [1:50:01<22:14, 667.30s/it]

R     = 0.563924 (95% CI: 0.558825, 0.569209)
RMSE  = 2.620773 (95% CI: 2.595202, 2.647393)
Model: Transformer



Transformer prepare: 100%|#############################################################| 18/18 [01:26<00:00,  5.23s/it]
                                                                                                                       
All models:  91%|############################################################9      | 10/11 [1:59:32<10:37, 637.41s/it]

R     = 0.498318 (95% CI: 0.492409, 0.504273)
RMSE  = 3.361104 (95% CI: 3.326433, 3.395933)
Model: Fusion



Fusion prepare: 100%|##################################################################| 18/18 [01:32<00:00,  4.31s/it]
                                                                                                                       
All models: 100%|###################################################################| 11/11 [2:07:51<00:00, 697.37s/it]

R     = 0.619598 (95% CI: 0.615156, 0.623774)
RMSE  = 2.396153 (95% CI: 2.372489, 2.420518)

Final summary
Origin             | R = 0.553945 [0.548736, 0.558635] | RMSE = 2.611996 [2.584399, 2.635659]
Prior              | R = 0.592255 [0.588419, 0.596360] | RMSE = 2.491589 [2.464664, 2.516933]
Only               | R = 0.552081 [0.546759, 0.557111] | RMSE = 2.597996 [2.570941, 2.622944]
P_E                | R = 0.600034 [0.595292, 0.604543] | RMSE = 2.506312 [2.481856, 2.528814]
Lr_0.000001        | R = 0.549843 [0.544621, 0.554636] | RMSE = 2.615734 [2.589222, 2.640653]
6_times            | R = 0.564422 [0.559206, 0.569293] | RMSE = 2.578681 [2.552353, 2.603905]
CDF                | R = 0.552693 [0.546702, 0.558153] | RMSE = 2.947154 [2.919118, 2.974691]
UNet               | R = 0.559096 [0.552879, 0.564566] | RMSE = 2.606955 [2.581351, 2.631821]
CNN-Transformer    | R = 0.563924 [0.558825, 0.569209] | RMSE = 2.620773 [2.595202, 2.647393]
Transformer        | R = 0.498318 [0.492409, 0.

In [23]:
import numpy as np
from tqdm import tqdm

# =========================================================
# A) 单个 pair 的 RMSPE 点估计
#    保持你的定义：RMSPE = RMSE / (max(testy) - min(testy))
#    这里 max/min 是原始 test_flat 沿 time 维计算
# =========================================================
def calc_pair_rmspe_point_estimate_from_flat(pred_flat, test_flat):
    pred_flat = np.asarray(pred_flat, dtype=np.float32)
    test_flat = np.asarray(test_flat, dtype=np.float32)

    n = pred_flat.shape[0]

    err2 = (pred_flat - test_flat) * (pred_flat - test_flat)
    rmse_pix = np.sqrt(np.sum(err2, axis=0, dtype=np.float64) / n)

    test_range_pix = (
        np.max(test_flat, axis=0).astype(np.float64)
        - np.min(test_flat, axis=0).astype(np.float64)
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        rmspe_pix = rmse_pix / test_range_pix

    rmspe_pix[~np.isfinite(rmspe_pix)] = np.nan
    rmspe_mean = float(np.nanmean(rmspe_pix))

    return rmspe_mean


# =========================================================
# B) 单个 pair 的 RMSPE bootstrap（快速版）
#    仍然使用 count_mat，不重新构造 bootstrap 样本
#    分母固定为原始 test_flat 的 max-min，和你之前定义一致
# =========================================================
def bootstrap_pair_rmspe_from_flat_fast(
    pred_flat,
    test_flat,
    count_mat,
    batch_boot=16,
    show_progress=False,
    desc="bootstrap_rmspe"
):
    pred_flat = np.asarray(pred_flat, dtype=np.float32)
    test_flat = np.asarray(test_flat, dtype=np.float32)
    count_mat = np.asarray(count_mat, dtype=np.float32)

    T, P = pred_flat.shape
    n_boot = count_mat.shape[0]
    n = float(T)

    ERR2 = (pred_flat - test_flat) * (pred_flat - test_flat)

    # 分母固定使用原始 test_flat 的 range
    test_range_pix = (
        np.max(test_flat, axis=0).astype(np.float64)
        - np.min(test_flat, axis=0).astype(np.float64)
    )
    valid_range = np.isfinite(test_range_pix) & (test_range_pix > 0)

    rmspe_boot = np.full(n_boot, np.nan, dtype=np.float64)

    starts = list(range(0, n_boot, batch_boot))
    iterator = starts
    if show_progress:
        iterator = tqdm(starts, desc=desc, leave=False)

    for start in iterator:
        end = min(start + batch_boot, n_boot)
        W = count_mat[start:end, :]   # (bs, T)

        # bootstrap 下的误差平方和
        Serr2 = W @ ERR2              # (bs, P)

        # bootstrap RMSE
        rmse_pix = np.sqrt(Serr2 / n) # (bs, P)

        # RMSPE = RMSE / 原始范围
        rmspe_pix = np.full_like(rmse_pix, np.nan, dtype=np.float64)
        rmspe_pix[:, valid_range] = (
            rmse_pix[:, valid_range] / test_range_pix[valid_range][None, :]
        )

        rmspe_boot[start:end] = np.nanmean(rmspe_pix, axis=1)

    return rmspe_boot


# =========================================================
# C) 追加计算 RMSPE 及其 95% CI
#    直接复用你前面已有的:
#    - namespace
#    - MODEL_CONFIG
#    - COMPONENTS
#    - LAGS
#    - N_BOOT
#    - BLOCK_SIZE
#    - BATCH_BOOT
#    - BASE_SEED
#    - SHOW_BOOT_BAR
#    - all_results
#    - get_var_from_namespace
#    - prepare_pair_flat
#    - build_bootstrap_count_matrix
# =========================================================
print("\n" + "=" * 90)
print("Append RMSPE 95% CI")
print("=" * 90)

model_iter_rmspe = tqdm(MODEL_CONFIG, desc="RMSPE models", dynamic_ncols=True, ascii=True)

for model_idx, (model_name, prefix_map) in enumerate(model_iter_rmspe):
    print("-" * 90)
    print(f"Model: {model_name}")

    combo_list = [(comp, lag) for comp in COMPONENTS for lag in LAGS]

    flat_pairs = []
    point_rmspe_list = []

    prepare_bar = tqdm(
        combo_list,
        desc=f"{model_name} RMSPE prepare",
        leave=False,
        dynamic_ncols=True,
        ascii=True
    )

    for comp, lag in prepare_bar:
        prefix = prefix_map[comp]

        pred_name = f"{prefix}_predicty_{comp}_{lag}"
        test_name = f"{prefix}_testy_{comp}_{lag}"

        pred = get_var_from_namespace(pred_name, namespace)
        test = get_var_from_namespace(test_name, namespace)

        pred_flat, test_flat = prepare_pair_flat(pred, test)
        flat_pairs.append((comp, lag, pred_flat, test_flat))

        rmspe_est = calc_pair_rmspe_point_estimate_from_flat(pred_flat, test_flat)
        point_rmspe_list.append(rmspe_est)

    # 模型层点估计：18 个组合平均
    model_RMSPE_est = float(np.nanmean(point_rmspe_list))

    # 为不同 T 建立 bootstrap count matrix 缓存
    count_cache = {}
    for _, _, pred_flat, _ in flat_pairs:
        T_here = pred_flat.shape[0]
        if T_here not in count_cache:
            seed_here = BASE_SEED + model_idx * 10000 + T_here
            count_cache[T_here] = build_bootstrap_count_matrix(
                T=T_here,
                n_boot=N_BOOT,
                block_size=BLOCK_SIZE,
                seed=seed_here
            )

    # 模型 bootstrap 累加
    model_RMSPE_boot_sum = np.zeros(N_BOOT, dtype=np.float64)

    pair_bar = tqdm(
        flat_pairs,
        desc=f"{model_name} RMSPE pairs",
        leave=False,
        dynamic_ncols=True,
        ascii=True
    )

    for comp, lag, pred_flat, test_flat in pair_bar:
        pair_bar.set_postfix_str(f"{comp}_{lag}")

        T_here = pred_flat.shape[0]
        count_mat = count_cache[T_here]

        rmspe_boot = bootstrap_pair_rmspe_from_flat_fast(
            pred_flat=pred_flat,
            test_flat=test_flat,
            count_mat=count_mat,
            batch_boot=BATCH_BOOT,
            show_progress=SHOW_BOOT_BAR,
            desc=f"{model_name} {comp}_{lag} RMSPE bootstrap"
        )

        model_RMSPE_boot_sum += rmspe_boot

    model_RMSPE_boot = model_RMSPE_boot_sum / len(flat_pairs)

    valid_boot = model_RMSPE_boot[np.isfinite(model_RMSPE_boot)]
    if valid_boot.size == 0:
        RMSPE_ci_lower, RMSPE_ci_upper = np.nan, np.nan
    else:
        RMSPE_ci_lower, RMSPE_ci_upper = np.percentile(valid_boot, [2.5, 97.5])

    # 写回原 all_results
    if model_name not in all_results:
        all_results[model_name] = {}

    all_results[model_name]["RMSPE_est"] = model_RMSPE_est
    all_results[model_name]["RMSPE_ci"] = (float(RMSPE_ci_lower), float(RMSPE_ci_upper))

    print(
        f"RMSPE = {model_RMSPE_est:.6f} "
        f"(95% CI: {RMSPE_ci_lower:.6f}, {RMSPE_ci_upper:.6f})"
    )

# =========================================================
# D) 更新最终汇总输出
# =========================================================
print("\n" + "=" * 90)
print("Final summary with RMSPE")
print("=" * 90)

for model_name, res in all_results.items():
    r_est = res.get("R_est", np.nan)
    r_ci = res.get("R_ci", (np.nan, np.nan))

    rmse_est = res.get("RMSE_est", np.nan)
    rmse_ci = res.get("RMSE_ci", (np.nan, np.nan))

    rmspe_est = res.get("RMSPE_est", np.nan)
    rmspe_ci = res.get("RMSPE_ci", (np.nan, np.nan))

    print(
        f"{model_name:<18s} | "
        f"R = {r_est:.6f} "
        f"[{r_ci[0]:.6f}, {r_ci[1]:.6f}] | "
        f"RMSE = {rmse_est:.6f} "
        f"[{rmse_ci[0]:.6f}, {rmse_ci[1]:.6f}] | "
        f"RMSPE = {rmspe_est:.6f} "
        f"[{rmspe_ci[0]:.6f}, {rmspe_ci[1]:.6f}]"
    )


UNet RMSPE prepare: 100%|##############################################################| 18/18 [01:08<00:00,  4.04s/it]
                                                                                                                       
RMSPE models:  73%|#################################################4                  | 8/11 [26:50<10:11, 203.69s/it]

RMSPE = 0.084772 (95% CI: 0.084033, 0.085432)
------------------------------------------------------------------------------------------
Model: CNN-Transformer



CNN-Transformer RMSPE prepare: 100%|###################################################| 18/18 [01:16<00:00,  4.66s/it]
                                                                                                                       
RMSPE models:  82%|#######################################################6            | 9/11 [31:02<07:17, 218.61s/it]

RMSPE = 0.084391 (95% CI: 0.083564, 0.085151)
------------------------------------------------------------------------------------------
Model: Transformer



Transformer RMSPE prepare: 100%|#######################################################| 18/18 [01:20<00:00,  4.62s/it]
                                                                                                                       
RMSPE models:  91%|############################################################9      | 10/11 [35:08<03:47, 227.21s/it]

RMSPE = 0.102519 (95% CI: 0.101621, 0.103380)
------------------------------------------------------------------------------------------
Model: Fusion



Fusion RMSPE prepare: 100%|############################################################| 18/18 [00:56<00:00,  2.19s/it]
                                                                                                                       
RMSPE models: 100%|###################################################################| 11/11 [39:05<00:00, 213.26s/it]

RMSPE = 0.077521 (95% CI: 0.076819, 0.078250)

Final summary with RMSPE
Origin             | R = 0.553945 [0.548736, 0.558635] | RMSE = 2.611996 [2.584399, 2.635659] | RMSPE = 0.084443 [0.083665, 0.085146]
Prior              | R = 0.592255 [0.588419, 0.596360] | RMSE = 2.491589 [2.464664, 2.516933] | RMSPE = 0.080382 [0.079525, 0.081108]
Only               | R = 0.552081 [0.546759, 0.557111] | RMSE = 2.597996 [2.570941, 2.622944] | RMSPE = 0.084215 [0.083400, 0.084906]
P_E                | R = 0.600034 [0.595292, 0.604543] | RMSE = 2.506312 [2.481856, 2.528814] | RMSPE = 0.080805 [0.080068, 0.081464]
Lr_0.000001        | R = 0.549843 [0.544621, 0.554636] | RMSE = 2.615734 [2.589222, 2.640653] | RMSPE = 0.084600 [0.083807, 0.085305]
6_times            | R = 0.564422 [0.559206, 0.569293] | RMSE = 2.578681 [2.552353, 2.603905] | RMSPE = 0.083378 [0.082623, 0.084099]
CDF                | R = 0.552693 [0.546702, 0.558153] | RMSE = 2.947154 [2.919118, 2.974691] | RMSPE = 0.096862 [0.096002, 